<img src="https://static.igem.wiki/teams/6333/wiki/illustrations/logos/sharp/logo.svg" height="200" align="right" style="height:240px">

# S(H)ARP

**S(H)ARP** — *Streptomyces* Hidden Antibiotic Regulated Pathways

Easy-to-use discovery of silent biosynthetic pathways in *Streptomyces* genomes. S(H)ARP reads a genome, follows the regulators that switch each pathway on or off, and maps candidate hidden pathways from end to end — pointing to targets you can later wake up with CRISPR activation.

Annotation is generated with [Bakta](https://github.com/oschwengers/bakta), motif scanning with [FIMO / MEME Suite](https://meme-suite.org/), and domain and neighborhood analysis through the [Rotifer](https://github.com/leepusp/rotifer) pipeline.

For more details, see the [S(H)ARP project wiki](https://2026.igem.wiki/usp-brazil/) and the team [background page](https://2026.igem.wiki/usp-brazil/background).

---

**Instructions**

1. Upload your input in **Cell 1** — a genome FASTA (genome-only mode, annotation runs automatically) or an annotated genome package (FASTA + GFF/GenBank + protein FASTA).
2. Run **Cell 2** to execute the analysis and download the results (`.zip` with tables and an HTML report).

Runtime note: genome-only mode installs Bakta and its database on first run, which takes several minutes. Later runs in the same session reuse what is already prepared.

---

*Developed by [iGEM USP-Brazil 2026](https://2026.igem.wiki/usp-brazil/). Contact: igem.uspbrasil@usp.br · [@igemuspbr](https://www.instagram.com/igemuspbr)*


In [ ]:
# @title S(H)ARP — Setup, upload input, and prepare backend { display-mode: "form" }
# @markdown Prepare Colab runtime, clone GitLab S(H)ARP, clone Rotifer, upload files, infer input mode automatically, and prepare Bakta only when needed.

JOB_NAME = "sharp_run_01" # @param {type:"string"}
ORGANISM_NAME = "Streptomyces sp." # @param {type:"string"}
STRAIN_NAME = "unknown" # @param {type:"string"}
SHOW_SETUP_LOGS = False # @param {type:"boolean"}

from pathlib import Path
import os
import sys
import re
import json
import time
import shutil
import hashlib
import subprocess
import zipfile
import tarfile
import gzip
import urllib.request
import importlib

os.environ["MPLBACKEND"] = "Agg"

# ============================================================
# Configuration
# ============================================================

SHARP_PROJECT_NAME = "sharp_igem_usp_brazil_2026"

SHARP_REPO_URL = "https://gitlab.igem.org/2026/software/usp-brazil/sharp.git"
SHARP_BRANCH = "main"

ROTIFER_REPO_URL = "https://github.com/leepusp/rotifer.git"
ROTIFER_BRANCH = "master"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()


def clean_name(value):
    value = str(value).strip() or "sharp_run_01"
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("._-")
    return value or "sharp_run_01"


RUN_NAME = clean_name(JOB_NAME)

# ============================================================
# Runtime paths
# ============================================================

PROJECT_DIR = CONTENT_DIR / SHARP_PROJECT_NAME
DATA_DIR = PROJECT_DIR / "data"
INPUT_DIR = DATA_DIR / "input"
NORMALIZED_DIR = DATA_DIR / "normalized"
DATABASES_DIR = PROJECT_DIR / "databases"
CONFIG_DIR = PROJECT_DIR / "config"
RESULTS_DIR = PROJECT_DIR / "results"
RUN_DIR = RESULTS_DIR / RUN_NAME
LOG_DIR = RUN_DIR / "logs"
TABLE_DIR = RUN_DIR / "tables"
REPORT_DIR = RUN_DIR / "report"

ENV_DIR = PROJECT_DIR / "envs"
MEME_ENV_DIR = ENV_DIR / "meme"
BAKTA_ENV_DIR = ENV_DIR / "bakta"

SHARP_DIR = CONTENT_DIR / "sharp"
ROTIFER_DIR = CONTENT_DIR / "rotifer"
ROTIFER_LIB = ROTIFER_DIR / "lib"
ROTIFER_BIN = ROTIFER_DIR / "bin"

COMMON_READY = PROJECT_DIR / "SHARP_COMMON_READY"

for directory in [
    PROJECT_DIR,
    DATA_DIR,
    INPUT_DIR,
    NORMALIZED_DIR,
    DATABASES_DIR,
    CONFIG_DIR,
    RESULTS_DIR,
    RUN_DIR,
    LOG_DIR,
    TABLE_DIR,
    REPORT_DIR,
    ENV_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SETUP_EVENTS = []

# ============================================================
# Small infrastructure helpers
# ============================================================

def log_event(message):
    event = {
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "message": str(message),
    }

    SETUP_EVENTS.append(event)

    if SHOW_SETUP_LOGS:
        print(message)


def run(command, check=True, env=None):
    command_env = os.environ.copy()
    command_env["MPLBACKEND"] = "Agg"

    if env:
        command_env.update(env)

    result = subprocess.run(
        command,
        shell=isinstance(command, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
        env=command_env,
    )

    if check and result.returncode != 0:
        command_text = " ".join(command) if isinstance(command, list) else command

        raise RuntimeError(
            "Command failed:\n"
            + command_text
            + "\n\nSTDOUT:\n"
            + (result.stdout or "")[-4000:]
            + "\n\nSTDERR:\n"
            + (result.stderr or "")[-4000:]
        )

    return result


def prepend_path(path):
    path = Path(path)

    if path.exists() and str(path) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = str(path) + os.pathsep + os.environ.get("PATH", "")


def prepend_pythonpath(path):
    path = Path(path)

    if not path.exists():
        return

    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

    current = os.environ.get("PYTHONPATH", "")
    parts = [item for item in current.split(os.pathsep) if item]

    if str(path) not in parts:
        os.environ["PYTHONPATH"] = str(path) + (os.pathsep + current if current else "")


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )


def file_md5(path):
    h = hashlib.md5()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024 * 8), b""):
            h.update(block)

    return h.hexdigest()


def file_sha256(path):
    h = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024 * 8), b""):
            h.update(block)

    return h.hexdigest()


def download_file(url, output_path):
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if output_path.exists() and output_path.stat().st_size > 0:
        return "existing"

    if shutil.which("curl"):
        run([
            "curl",
            "-L",
            "--fail",
            "--retry",
            "5",
            "--retry-delay",
            "5",
            url,
            "-o",
            str(output_path),
        ])
    else:
        with urllib.request.urlopen(url, timeout=120) as response:
            output_path.write_bytes(response.read())

    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"Downloaded file is empty: {url}")

    return url


def open_text_maybe_gzip(path):
    path = Path(path)

    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")

    return path.open("r", encoding="utf-8", errors="replace")


def copy_maybe_decompress(source_path, destination_path):
    source_path = Path(source_path)
    destination_path = Path(destination_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if source_path.name.lower().endswith(".gz"):
        with gzip.open(source_path, "rb") as src, destination_path.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    else:
        shutil.copy2(source_path, destination_path)

    return destination_path


def fasta_stats(path):
    records = 0
    total = 0
    current = 0
    first_id = ""

    with open_text_maybe_gzip(path) as handle:
        for line in handle:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if records:
                    total += current

                records += 1
                current = 0

                if not first_id:
                    first_id = line[1:].split()[0]
            else:
                current += len(re.sub(r"[^A-Za-z]", "", line))

    if records:
        total += current

    return {
        "records": records,
        "total_bp": total,
        "first_id": first_id,
    }


def safe_extract_zip(archive_path, output_dir):
    archive_path = Path(archive_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_root = output_dir.resolve()

    extracted = []

    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.namelist():
            target = (output_dir / member).resolve()

            try:
                target.relative_to(output_root)
            except ValueError:
                raise RuntimeError(f"Unsafe ZIP member path: {member}")

        archive.extractall(output_dir)

    extracted.extend([path for path in output_dir.rglob("*") if path.is_file()])

    return extracted


def safe_extract_tar(archive_path, output_dir):
    archive_path = Path(archive_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_root = output_dir.resolve()

    extracted = []

    with tarfile.open(archive_path, "r:*") as archive:
        members = archive.getmembers()

        for member in members:
            target = (output_dir / member.name).resolve()

            try:
                target.relative_to(output_root)
            except ValueError:
                raise RuntimeError(f"Unsafe TAR member path: {member.name}")

        try:
            archive.extractall(output_dir, filter="data")
        except TypeError:
            archive.extractall(output_dir)

    extracted.extend([path for path in output_dir.rglob("*") if path.is_file()])

    return extracted


def safe_extract_tar_xz(archive_path, output_dir):
    archive_path = Path(archive_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_root = output_dir.resolve()

    with tarfile.open(archive_path, "r:xz") as archive:
        members = archive.getmembers()

        for member in members:
            target = (output_dir / member.name).resolve()

            try:
                target.relative_to(output_root)
            except ValueError:
                raise RuntimeError(f"Unsafe TAR member path: {member.name}")

        try:
            archive.extractall(output_dir, filter="data")
        except TypeError:
            archive.extractall(output_dir)


def ensure_repo(repo_url, branch, target_dir):
    target_dir = Path(target_dir)

    if (target_dir / ".git").exists():
        run(["git", "-C", str(target_dir), "fetch", "origin", branch, "--depth", "1"])
        run(["git", "-C", str(target_dir), "checkout", branch])
        run(["git", "-C", str(target_dir), "reset", "--hard", f"origin/{branch}"])
    else:
        if target_dir.exists():
            shutil.rmtree(target_dir)

        run([
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            branch,
            repo_url,
            str(target_dir),
        ])

    return target_dir


def ensure_mamba():
    prepend_path("/usr/local/bin")

    mamba = shutil.which("mamba")

    if mamba:
        return mamba

    installer = ENV_DIR / "Miniforge3-Linux-x86_64.sh"

    if not installer.exists():
        run([
            "wget",
            "-q",
            "-O",
            str(installer),
            "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh",
        ])

    run(["bash", str(installer), "-bfp", "/usr/local"])
    run(["mamba", "config", "--set", "auto_update_conda", "false"], check=False)

    mamba = shutil.which("mamba")

    if not mamba:
        raise RuntimeError("mamba was not found after Miniforge installation.")

    return mamba


def hmmpress_local(hmm_path):
    """Create HMMER binary indexes for a local HMM file."""
    hmm_path = Path(hmm_path)

    if not hmm_path.exists() or hmm_path.stat().st_size == 0:
        raise RuntimeError(f"HMM file not found or empty: {hmm_path}")

    for ext in [".h3f", ".h3i", ".h3m", ".h3p"]:
        index_path = Path(str(hmm_path) + ext)

        if index_path.exists():
            index_path.unlink()

    run(["hmmpress", "-f", str(hmm_path)], check=True)

    expected = [Path(str(hmm_path) + ext) for ext in [".h3f", ".h3i", ".h3m", ".h3p"]]
    missing = [str(path) for path in expected if not path.exists()]

    if missing:
        raise RuntimeError("hmmpress did not create all expected index files: " + ", ".join(missing))

    return {
        "hmm": str(hmm_path),
        "indexes": [str(path) for path in expected],
    }


# ============================================================
# Common dependencies
# ============================================================

print(f"S(H)ARP setup started: {RUN_NAME}")

prepend_path(MEME_ENV_DIR / "bin")
prepend_path(BAKTA_ENV_DIR / "bin")
prepend_path("/usr/local/bin")
prepend_path("/usr/bin")
prepend_path("/bin")

log_event("Preparing common runtime dependencies.")

if IN_COLAB and not COMMON_READY.exists():
    run("apt-get update -qq")

    run(
        "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
        "git curl wget xz-utils bzip2 hmmer graphviz graphviz-dev pkg-config",
        check=False,
    )

    run([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "numpy",
        "pandas",
        "biopython",
        "bcbio-gff",
        "requests",
        "tqdm",
        "joblib",
        "pyhmmer",
        "ete3",
        "seaborn",
        "matplotlib",
        "networkx",
        "sqlalchemy",
        "openpyxl",
        "pyyaml",
    ])

    try:
        run([sys.executable, "-m", "pip", "install", "-q", "pygraphviz"], check=True)
    except Exception:
        pass

    COMMON_READY.touch()

try:
    from BCBio import GFF  # noqa: F401
except Exception:
    run([sys.executable, "-m", "pip", "install", "-q", "bcbio-gff"])
    importlib.invalidate_caches()
    from BCBio import GFF  # noqa: F401

# ============================================================
# Clone GitLab S(H)ARP package and Rotifer runtime dependency
# ============================================================

log_event("Cloning GitLab S(H)ARP package.")
ensure_repo(SHARP_REPO_URL, SHARP_BRANCH, SHARP_DIR)

log_event("Cloning Rotifer runtime dependency.")
ensure_repo(ROTIFER_REPO_URL, ROTIFER_BRANCH, ROTIFER_DIR)

prepend_pythonpath(ROTIFER_LIB)
prepend_pythonpath(SHARP_DIR)
prepend_path(ROTIFER_BIN)

# ============================================================
# Prepare FIMO/MEME
# ============================================================

log_event("Preparing FIMO/MEME.")

if not shutil.which("fimo"):
    mamba = ensure_mamba()

    if not (MEME_ENV_DIR / "bin" / "fimo").exists():
        run([
            mamba,
            "create",
            "-y",
            "-p",
            str(MEME_ENV_DIR),
            "-c",
            "conda-forge",
            "-c",
            "bioconda",
            "meme",
        ])

    prepend_path(MEME_ENV_DIR / "bin")

if not shutil.which("fimo"):
    raise RuntimeError("FIMO was not found after MEME Suite installation.")

# ============================================================
# Validate official S(H)ARP package
# ============================================================

log_event("Validating S(H)ARP package.")

sharp = importlib.import_module("sharp")
sharp_pipeline = importlib.import_module("sharp.pipeline")

if not hasattr(sharp_pipeline, "igem_pipeline"):
    raise RuntimeError("sharp.pipeline.igem_pipeline was not found.")

SHARP_PACKAGE_STATUS = {
    "import_ok": True,
    "repo_url": SHARP_REPO_URL,
    "branch": SHARP_BRANCH,
    "sharp_dir": str(SHARP_DIR),
    "sharp_file": str(Path(sharp.__file__).resolve()),
    "pipeline_file": str(Path(sharp_pipeline.__file__).resolve()),
    "pipeline_function": "sharp.pipeline.igem_pipeline",
}

# ============================================================
# Copy Colab resources from GitLab clone
# ============================================================

log_event("Preparing S(H)ARP Colab resources.")

GITLAB_COLAB_RESOURCES = SHARP_DIR / "colab" / "resources"

RESOURCE_SOURCES = {
    "bakta_light_config": GITLAB_COLAB_RESOURCES / "config" / "bakta_light_cache.json",
    "heptamer_meme": GITLAB_COLAB_RESOURCES / "motifs" / "heptarepeats2.meme",
    "sarp_hmm": GITLAB_COLAB_RESOURCES / "hmm" / "sarp_custom.hmm",
    "domain_models_hmm": GITLAB_COLAB_RESOURCES / "domain_models" / "domain_models.hmm",
    "domain_modelnames": GITLAB_COLAB_RESOURCES / "domain_models" / "hmm_modelnames.tsv",
    "sharp_config": SHARP_DIR / "config.yaml",
}

SHARP_INTERNAL_RESOURCES = {
    "bakta_light_config": str(CONFIG_DIR / "bakta_light_cache.json"),
    "heptamer_meme": str(DATABASES_DIR / "motifs" / "heptarepeats2.meme"),
    "sarp_hmm": str(DATABASES_DIR / "hmm" / "sarp_custom.hmm"),
    "domain_models_hmm": str(DATABASES_DIR / "domain_models" / "domain_models.hmm"),
    "domain_modelnames": str(DATABASES_DIR / "domain_models" / "hmm_modelnames.tsv"),
    "sharp_config": str(CONFIG_DIR / "sharp_config.yaml"),
}

SHARP_RESOURCE_STATUS = {}

for key, source_path in RESOURCE_SOURCES.items():
    source_path = Path(source_path)
    target_path = Path(SHARP_INTERNAL_RESOURCES[key])
    target_path.parent.mkdir(parents=True, exist_ok=True)

    if not source_path.exists() or source_path.stat().st_size == 0:
        raise RuntimeError(f"Missing resource in GitLab S(H)ARP clone: {source_path}")

    shutil.copy2(source_path, target_path)

    SHARP_RESOURCE_STATUS[key] = {
        "available": True,
        "source": str(source_path),
        "path": str(target_path),
        "size": target_path.stat().st_size,
        "sha256": file_sha256(target_path),
    }

SHARP_HMM_PRESS_STATUS = [
    hmmpress_local(SHARP_INTERNAL_RESOURCES["domain_models_hmm"]),
    hmmpress_local(SHARP_INTERNAL_RESOURCES["sarp_hmm"]),
]

# ============================================================
# Upload input files and infer mode automatically
# ============================================================

log_event("Uploading and normalizing input files.")

if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)

if NORMALIZED_DIR.exists():
    shutil.rmtree(NORMALIZED_DIR)

INPUT_DIR.mkdir(parents=True, exist_ok=True)
NORMALIZED_DIR.mkdir(parents=True, exist_ok=True)

uploaded_paths = []

if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()

    for filename, content in uploaded.items():
        destination = INPUT_DIR / Path(filename).name
        destination.write_bytes(content)
        uploaded_paths.append(destination)
else:
    uploaded_paths = [path for path in INPUT_DIR.rglob("*") if path.is_file()]


def safe_extract(path):
    path = Path(path)
    lower = path.name.lower()
    extracted = []

    if lower.endswith(".zip"):
        target = INPUT_DIR / f"{path.stem}_extracted"
        extracted.extend(safe_extract_zip(path, target))

    elif lower.endswith((".tar", ".tar.gz", ".tgz", ".tar.xz")):
        target = INPUT_DIR / f"{path.name.replace('.', '_')}_extracted"
        extracted.extend(safe_extract_tar(path, target))

    return extracted


for path in list(uploaded_paths):
    uploaded_paths.extend(safe_extract(path))

all_files = [path for path in INPUT_DIR.rglob("*") if path.is_file()]

if not all_files:
    raise RuntimeError("No input files were uploaded.")


def normalized_name(path):
    name = Path(path).name.lower()

    if name.endswith(".gz"):
        name = name[:-3]

    return name


def suffix(path):
    return Path(normalized_name(path)).suffix.lower()


def is_annotation(path):
    return suffix(path) in {".gff", ".gff3", ".gb", ".gbk", ".gbff", ".genbank"}


def is_protein_fasta(path):
    name = normalized_name(path)

    return (
        name.endswith(".faa")
        or name.endswith(".pep")
        or "protein" in name
        or "proteins" in name
    )


def is_cds_fasta(path):
    name = normalized_name(path)

    return name.endswith(".ffn") or "cds" in name


def is_fasta(path):
    return suffix(path) in {".fa", ".fas", ".fasta", ".fna", ".ffn", ".faa", ".pep"}


def is_genome_fasta(path):
    return is_fasta(path) and not is_protein_fasta(path) and not is_cds_fasta(path)


def choose_largest_fasta(paths):
    if not paths:
        return None

    return sorted(
        paths,
        key=lambda item: fasta_stats(item)["total_bp"],
        reverse=True,
    )[0]


genome_candidates = [path for path in all_files if is_genome_fasta(path)]
annotation_candidates = [path for path in all_files if is_annotation(path)]
protein_candidates = [path for path in all_files if is_protein_fasta(path)]
cds_candidates = [path for path in all_files if is_cds_fasta(path)]

genome_source = choose_largest_fasta(genome_candidates)
annotation_source = annotation_candidates[0] if annotation_candidates else None
protein_source = protein_candidates[0] if protein_candidates else None
cds_source = cds_candidates[0] if cds_candidates else None

if genome_source is None:
    detected_names = ", ".join(path.name for path in all_files)
    raise RuntimeError(f"No genome FASTA was detected. Detected files: {detected_names}")

has_partial_annotation_package = annotation_source is not None or protein_source is not None

if annotation_source is not None and protein_source is not None:
    INPUT_MODE_KEY = "annotated_genome_package"
    workflow = "sharp_existing_annotation"
    annotation_mode = "use_existing_annotation"
    annotation_backend = "existing_annotation"
    requires_bakta_execution = False
    requires_bakta_db = False
elif has_partial_annotation_package:
    detected_names = ", ".join(path.name for path in all_files)

    raise RuntimeError(
        "Partial annotated package detected. Provide genome FASTA + annotation file + protein FASTA, "
        f"or provide only genome FASTA for automatic Bakta annotation. Detected files: {detected_names}"
    )
else:
    INPUT_MODE_KEY = "genome_fasta_only"
    workflow = "sharp_auto_from_genome"
    annotation_mode = "auto_from_genome"
    annotation_backend = "bakta_cached_light_db"
    requires_bakta_execution = True
    requires_bakta_db = True

GENOME_FASTA = NORMALIZED_DIR / "sharp_input_genome.fna"
ANNOTATION_FILE = None
PROTEIN_FASTA = None
CDS_FASTA = None
genome_format = ""

copy_maybe_decompress(genome_source, GENOME_FASTA)

if INPUT_MODE_KEY == "annotated_genome_package":
    if suffix(annotation_source) in {".gb", ".gbk", ".gbff", ".genbank"}:
        ANNOTATION_FILE = NORMALIZED_DIR / "sharp_input_annotation.gbff"
        genome_format = "gbk"
    else:
        ANNOTATION_FILE = NORMALIZED_DIR / "sharp_input_annotation.gff3"
        genome_format = "gff"

    PROTEIN_FASTA = NORMALIZED_DIR / "sharp_input_proteins.faa"

    copy_maybe_decompress(annotation_source, ANNOTATION_FILE)
    copy_maybe_decompress(protein_source, PROTEIN_FASTA)

    if cds_source is not None:
        CDS_FASTA = NORMALIZED_DIR / "sharp_input_cds.ffn"
        copy_maybe_decompress(cds_source, CDS_FASTA)

genome_stats = fasta_stats(GENOME_FASTA)
genome_hash = file_sha256(GENOME_FASTA)

# ============================================================
# Prepare Bakta backend only for genome FASTA only mode
# ============================================================

def ensure_bakta_backend():
    log_event("Preparing Bakta backend.")

    prepend_path(BAKTA_ENV_DIR / "bin")

    bakta_bin = shutil.which("bakta")

    if not bakta_bin:
        mamba = ensure_mamba()

        if not (BAKTA_ENV_DIR / "bin" / "bakta").exists():
            run([
                mamba,
                "create",
                "-y",
                "-p",
                str(BAKTA_ENV_DIR),
                "-c",
                "conda-forge",
                "-c",
                "bioconda",
                "bakta=1.11.3",
                "amrfinderplus",
            ])

        prepend_path(BAKTA_ENV_DIR / "bin")
        bakta_bin = shutil.which("bakta")

    if not bakta_bin:
        raise RuntimeError("Bakta was not found after installation.")

    bakta_config_path = Path(SHARP_INTERNAL_RESOURCES["bakta_light_config"])

    if not bakta_config_path.exists():
        raise RuntimeError(f"Bakta light config not found: {bakta_config_path}")

    bakta_config = json.loads(bakta_config_path.read_text(encoding="utf-8"))
    bakta_section = bakta_config.get("bakta", bakta_config)

    archive_name = bakta_section.get("archive_name", "db-light.tar.xz")
    expected_md5 = bakta_section.get("expected_md5", "")
    expected_sha256 = bakta_section.get("expected_sha256", "")
    release_url = bakta_section.get("github_release_url", "")
    fallback_url = bakta_section.get("zenodo_fallback_url", "")

    if not release_url:
        raise RuntimeError("Bakta light config does not define github_release_url.")

    bakta_cache_dir = DATABASES_DIR / "bakta_cache"
    bakta_extract_dir = DATABASES_DIR / "bakta"
    bakta_archive = bakta_cache_dir / archive_name
    bakta_db_path = bakta_extract_dir / "db-light"

    bakta_cache_dir.mkdir(parents=True, exist_ok=True)
    bakta_extract_dir.mkdir(parents=True, exist_ok=True)

    archive_ready = bakta_archive.exists() and bakta_archive.stat().st_size > 0

    if archive_ready and expected_md5:
        archive_ready = file_md5(bakta_archive) == expected_md5

    if archive_ready and expected_sha256:
        archive_ready = file_sha256(bakta_archive) == expected_sha256

    if not archive_ready:
        if bakta_archive.exists():
            bakta_archive.unlink()

        try:
            download_file(release_url, bakta_archive)
        except Exception:
            if not fallback_url:
                raise

            download_file(fallback_url, bakta_archive)

    if expected_md5 and file_md5(bakta_archive) != expected_md5:
        raise RuntimeError("Bakta light DB archive MD5 mismatch.")

    if expected_sha256 and file_sha256(bakta_archive) != expected_sha256:
        raise RuntimeError("Bakta light DB archive SHA256 mismatch.")

    if not (bakta_db_path / "version.json").exists():
        safe_extract_tar_xz(bakta_archive, bakta_extract_dir)

    if not (bakta_db_path / "version.json").exists():
        version_files = sorted(bakta_extract_dir.rglob("version.json"))

        if version_files:
            bakta_db_path = version_files[0].parent

    if not (bakta_db_path / "version.json").exists():
        raise RuntimeError("Bakta light DB extraction failed: version.json not found.")

    bakta_db_version = json.loads((bakta_db_path / "version.json").read_text(encoding="utf-8"))

    os.environ["BAKTA_DB"] = str(bakta_db_path)

    amrfinder_update = shutil.which("amrfinder_update")

    if not amrfinder_update:
        raise RuntimeError("amrfinder_update was not found after Bakta installation.")

    amrfinder_db_path = bakta_db_path / "amrfinderplus-db"
    amrfinder_sentinel = amrfinder_db_path / ".sharp_amrfinder_update_ok.json"

    if not amrfinder_sentinel.exists():
        run([
            amrfinder_update,
            "--force_update",
            "--database",
            str(amrfinder_db_path),
        ])

        write_json(
            amrfinder_sentinel,
            {
                "status": "ready",
                "database": str(amrfinder_db_path),
                "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            },
        )

    bakta_threads = min(2, max(1, os.cpu_count() or 2))

    bakta_runtime_args = [
        "--threads",
        str(bakta_threads),
        "--skip-trna",
        "--skip-tmrna",
        "--skip-rrna",
        "--skip-ncrna",
        "--skip-ncrna-region",
        "--skip-crispr",
        "--skip-sorf",
        "--skip-gap",
        "--skip-ori",
        "--skip-plot",
    ]

    return {
        "requires_bakta_execution": True,
        "requires_bakta_db": True,
        "bakta_bin": str(bakta_bin),
        "bakta_db_ready": True,
        "bakta_db_path": str(bakta_db_path),
        "bakta_db_version": bakta_db_version,
        "bakta_threads": bakta_threads,
        "bakta_runtime_args": bakta_runtime_args,
        "amrfinder_db_ready": True,
        "amrfinder_db_path": str(amrfinder_db_path),
        "amrfinder_sentinel": str(amrfinder_sentinel),
    }


if requires_bakta_execution:
    SHARP_BACKEND = ensure_bakta_backend()
else:
    SHARP_BACKEND = {
        "requires_bakta_execution": False,
        "requires_bakta_db": False,
        "bakta_bin": "",
        "bakta_db_ready": False,
        "bakta_db_path": "",
        "bakta_db_version": {},
        "bakta_threads": 0,
        "bakta_runtime_args": [],
        "amrfinder_db_ready": False,
        "amrfinder_db_path": "",
    }

# ============================================================
# Save context for execution cell
# ============================================================

SHARP_CONTEXT_FILE = CONFIG_DIR / "sharp_notebook_context.json"
INPUT_CONTRACT_FILE = CONFIG_DIR / "sharp_input_contract.json"
INPUT_FILES_FILE = CONFIG_DIR / "sharp_input_files.json"
RESOURCE_MANIFEST_FILE = CONFIG_DIR / "sharp_resource_manifest.json"
DEPENDENCY_MANIFEST_FILE = RUN_DIR / f"{RUN_NAME}_dependency_manifest.json"
SETUP_LOG_FILE = RUN_DIR / f"{RUN_NAME}_setup_events.json"

SHARP_PATHS = {
    "project_dir": str(PROJECT_DIR),
    "data_dir": str(DATA_DIR),
    "input_dir": str(INPUT_DIR),
    "normalized_dir": str(NORMALIZED_DIR),
    "databases_dir": str(DATABASES_DIR),
    "config_dir": str(CONFIG_DIR),
    "results_dir": str(RESULTS_DIR),
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "log_dir": str(LOG_DIR),
    "table_dir": str(TABLE_DIR),
    "report_dir": str(REPORT_DIR),
    "sharp_dir": str(SHARP_DIR),
    "rotifer_dir": str(ROTIFER_DIR),
    "rotifer_lib": str(ROTIFER_LIB),
    "rotifer_bin": str(ROTIFER_BIN),
}

SHARP_INPUT_CONTRACT = {
    "status": "ready",
    "workflow": workflow,
    "mode": INPUT_MODE_KEY,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "requires_bakta_execution": requires_bakta_execution,
    "requires_bakta_db": requires_bakta_db,
    "genome_fasta": str(GENOME_FASTA),
    "annotation_file": str(ANNOTATION_FILE) if ANNOTATION_FILE else "",
    "genome_format": genome_format,
    "protein_fasta": str(PROTEIN_FASTA) if PROTEIN_FASTA else "",
    "cds_fasta": str(CDS_FASTA) if CDS_FASTA else "",
    "organism_name": ORGANISM_NAME,
    "strain_name": STRAIN_NAME,
    "genome_records": genome_stats["records"],
    "genome_total_bp": genome_stats["total_bp"],
    "genome_first_id": genome_stats["first_id"],
    "genome_sha256": genome_hash,
}

INPUT_FILES = {
    "uploaded_files": [str(path) for path in uploaded_paths],
    "all_detected_files": [str(path) for path in all_files],
    "genome_candidates": [str(path) for path in genome_candidates],
    "annotation_candidates": [str(path) for path in annotation_candidates],
    "protein_candidates": [str(path) for path in protein_candidates],
    "cds_candidates": [str(path) for path in cds_candidates],
    "genome_source": str(genome_source),
    "annotation_source": str(annotation_source) if annotation_source else "",
    "protein_source": str(protein_source) if protein_source else "",
    "cds_source": str(cds_source) if cds_source else "",
}

SHARP_NOTEBOOK_CONTEXT = {
    "project": SHARP_PROJECT_NAME,
    "initialized_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "job_name": RUN_NAME,
    "run_name": RUN_NAME,
    "workflow": workflow,
    "input_mode": INPUT_MODE_KEY,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "paths": SHARP_PATHS,
    "input_contract": SHARP_INPUT_CONTRACT,
    "input_files": INPUT_FILES,
    "sharp_package": SHARP_PACKAGE_STATUS,
    "internal_resources": SHARP_INTERNAL_RESOURCES,
    "resource_status": SHARP_RESOURCE_STATUS,
    "hmm_press_status": SHARP_HMM_PRESS_STATUS,
    "backend": SHARP_BACKEND,
    "fimo_bin": shutil.which("fimo") or "",
    "pythonpath": os.environ.get("PYTHONPATH", ""),
    "path": os.environ.get("PATH", ""),
}

SHARP_DEPENDENCY_STATUS = {
    "ready": True,
    "sharp_package": SHARP_PACKAGE_STATUS,
    "workflow": workflow,
    "input_mode": INPUT_MODE_KEY,
    "annotation_backend": annotation_backend,
    "backend": SHARP_BACKEND,
    "resource_status": SHARP_RESOURCE_STATUS,
    "hmm_press_status": SHARP_HMM_PRESS_STATUS,
    "setup_events": SETUP_EVENTS,
}

write_json(SHARP_CONTEXT_FILE, SHARP_NOTEBOOK_CONTEXT)
write_json(INPUT_CONTRACT_FILE, SHARP_INPUT_CONTRACT)
write_json(INPUT_FILES_FILE, INPUT_FILES)
write_json(RESOURCE_MANIFEST_FILE, SHARP_RESOURCE_STATUS)
write_json(DEPENDENCY_MANIFEST_FILE, SHARP_DEPENDENCY_STATUS)
write_json(SETUP_LOG_FILE, SETUP_EVENTS)

globals().update({
    "PROJECT_DIR": PROJECT_DIR,
    "DATA_DIR": DATA_DIR,
    "INPUT_DIR": INPUT_DIR,
    "NORMALIZED_DIR": NORMALIZED_DIR,
    "DATABASES_DIR": DATABASES_DIR,
    "CONFIG_DIR": CONFIG_DIR,
    "RESULTS_DIR": RESULTS_DIR,
    "RUN_NAME": RUN_NAME,
    "RUN_DIR": RUN_DIR,
    "LOG_DIR": LOG_DIR,
    "TABLE_DIR": TABLE_DIR,
    "REPORT_DIR": REPORT_DIR,
    "SHARP_DIR": SHARP_DIR,
    "ROTIFER_DIR": ROTIFER_DIR,
    "ROTIFER_LIB": ROTIFER_LIB,
    "ROTIFER_BIN": ROTIFER_BIN,
    "GENOME_FASTA": GENOME_FASTA,
    "ANNOTATION_FILE": ANNOTATION_FILE,
    "PROTEIN_FASTA": PROTEIN_FASTA,
    "CDS_FASTA": CDS_FASTA,
    "genome_format": genome_format,
    "workflow": workflow,
    "input_mode": INPUT_MODE_KEY,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "requires_bakta_execution": requires_bakta_execution,
    "requires_bakta_db": requires_bakta_db,
    "SHARP_INTERNAL_RESOURCES": SHARP_INTERNAL_RESOURCES,
    "SHARP_RESOURCE_STATUS": SHARP_RESOURCE_STATUS,
    "SHARP_BACKEND": SHARP_BACKEND,
    "SHARP_PACKAGE_STATUS": SHARP_PACKAGE_STATUS,
    "SHARP_HMM_PRESS_STATUS": SHARP_HMM_PRESS_STATUS,
    "SHARP_CONTEXT_FILE": SHARP_CONTEXT_FILE,
    "INPUT_CONTRACT_FILE": INPUT_CONTRACT_FILE,
    "INPUT_FILES_FILE": INPUT_FILES_FILE,
    "RESOURCE_MANIFEST_FILE": RESOURCE_MANIFEST_FILE,
    "DEPENDENCY_MANIFEST_FILE": DEPENDENCY_MANIFEST_FILE,
    "SETUP_LOG_FILE": SETUP_LOG_FILE,
    "BAKTA_BIN": SHARP_BACKEND.get("bakta_bin", ""),
    "BAKTA_DB_READY": SHARP_BACKEND.get("bakta_db_ready", False),
    "BAKTA_DB_PATH": SHARP_BACKEND.get("bakta_db_path", ""),
    "BAKTA_RUNTIME_ARGS": SHARP_BACKEND.get("bakta_runtime_args", []),
    "BAKTA_THREADS": SHARP_BACKEND.get("bakta_threads", 0),
})

print("S(H)ARP input ready")
print("job:", RUN_NAME)
print("mode:", INPUT_MODE_KEY)
print("genome:", f"{genome_stats['records']} records / {genome_stats['total_bp']:,} bp")
print("sharp:", "ready")
print("rotifer:", "ready")
print("resources:", "ready")
print("annotation_backend:", annotation_backend)
print("backend:", "ready" if (not requires_bakta_execution or SHARP_BACKEND.get("bakta_db_ready")) else "check")
print("next:", "run S(H)ARP GitLab pipeline")


In [ ]:
# @title S(H)ARP — Run GitLab pipeline and export results { display-mode: "form" }
# @markdown Run S(H)ARP by calling the official GitLab `sharp.pipeline.igem_pipeline` function.

DOWNLOAD_RESULTS_ZIP = True # @param {type:"boolean"}

from pathlib import Path
import os
import sys
import json
import time
import shutil
import zipfile
import importlib

os.environ["MPLBACKEND"] = "Agg"

# ============================================================
# Load context from Cell 1
# ============================================================

PROJECT_DIR = Path(globals().get("PROJECT_DIR", "/content/sharp_igem_usp_brazil_2026")).resolve()
CONFIG_DIR = Path(globals().get("CONFIG_DIR", PROJECT_DIR / "config")).resolve()

SHARP_CONTEXT_FILE = Path(globals().get("SHARP_CONTEXT_FILE", CONFIG_DIR / "sharp_notebook_context.json")).resolve()
INPUT_CONTRACT_FILE = Path(globals().get("INPUT_CONTRACT_FILE", CONFIG_DIR / "sharp_input_contract.json")).resolve()

if not SHARP_CONTEXT_FILE.exists():
    raise RuntimeError("Missing sharp_notebook_context.json. Run Cell 1 first.")

if not INPUT_CONTRACT_FILE.exists():
    raise RuntimeError("Missing sharp_input_contract.json. Run Cell 1 first.")

context = json.loads(SHARP_CONTEXT_FILE.read_text(encoding="utf-8"))
contract = json.loads(INPUT_CONTRACT_FILE.read_text(encoding="utf-8"))

RUN_NAME = context.get("run_name", context.get("job_name", "sharp_run_01"))
PATHS = context.get("paths", {})

RESULTS_DIR = Path(PATHS.get("results_dir", PROJECT_DIR / "results")).resolve()
RUN_DIR = Path(PATHS.get("run_dir", RESULTS_DIR / RUN_NAME)).resolve()
LOG_DIR = Path(PATHS.get("log_dir", RUN_DIR / "logs")).resolve()
TABLE_DIR = Path(PATHS.get("table_dir", RUN_DIR / "tables")).resolve()
REPORT_DIR = Path(PATHS.get("report_dir", RUN_DIR / "report")).resolve()

ANNOTATION_DIR = RUN_DIR / "annotation"
BAKTA_RUN_DIR = RUN_DIR / "bakta"

for directory in [
    RUN_DIR,
    LOG_DIR,
    TABLE_DIR,
    REPORT_DIR,
    ANNOTATION_DIR,
    BAKTA_RUN_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

GENOME_FASTA = Path(contract["genome_fasta"]).resolve()
ANNOTATION_FILE_FROM_CONTRACT = Path(contract["annotation_file"]).resolve() if contract.get("annotation_file") else None
PROTEIN_FASTA_FROM_CONTRACT = Path(contract["protein_fasta"]).resolve() if contract.get("protein_fasta") else None
CDS_FASTA_FROM_CONTRACT = Path(contract["cds_fasta"]).resolve() if contract.get("cds_fasta") else None

input_mode = contract["mode"]
requires_bakta_execution = bool(contract.get("requires_bakta_execution", input_mode == "genome_fasta_only"))

SHARP_BACKEND = globals().get("SHARP_BACKEND", context.get("backend", {}))
SHARP_INTERNAL_RESOURCES = globals().get("SHARP_INTERNAL_RESOURCES", context.get("internal_resources", {}))

SHARP_DIR = Path(PATHS.get("sharp_dir", "/content/sharp")).resolve()
ROTIFER_LIB = Path(PATHS.get("rotifer_lib", "/content/rotifer/lib")).resolve()
ROTIFER_BIN = Path(PATHS.get("rotifer_bin", "/content/rotifer/bin")).resolve()

if SHARP_DIR.exists() and str(SHARP_DIR) not in sys.path:
    sys.path.insert(0, str(SHARP_DIR))

if ROTIFER_LIB.exists() and str(ROTIFER_LIB) not in sys.path:
    sys.path.insert(0, str(ROTIFER_LIB))

if ROTIFER_BIN.exists() and str(ROTIFER_BIN) not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = str(ROTIFER_BIN) + os.pathsep + os.environ.get("PATH", "")

BAKTA_BIN = globals().get("BAKTA_BIN", SHARP_BACKEND.get("bakta_bin", "")) or shutil.which("bakta")
BAKTA_DB_PATH = globals().get("BAKTA_DB_PATH", SHARP_BACKEND.get("bakta_db_path", ""))
BAKTA_RUNTIME_ARGS = globals().get("BAKTA_RUNTIME_ARGS", SHARP_BACKEND.get("bakta_runtime_args", []))
BAKTA_THREADS = int(globals().get("BAKTA_THREADS", SHARP_BACKEND.get("bakta_threads", 2)) or 2)

if BAKTA_BIN:
    bakta_bin_dir = Path(BAKTA_BIN).parent

    if bakta_bin_dir.exists() and str(bakta_bin_dir) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = str(bakta_bin_dir) + os.pathsep + os.environ.get("PATH", "")

# ============================================================
# Import official GitLab SHARP package
# ============================================================

sharp = importlib.import_module("sharp")
sharp_pipeline = importlib.import_module("sharp.pipeline")
sharp_cli = importlib.import_module("sharp.cli")

if not hasattr(sharp_pipeline, "igem_pipeline"):
    raise RuntimeError("sharp.pipeline.igem_pipeline was not found.")

if requires_bakta_execution and not hasattr(sharp_cli, "run_bakta"):
    raise RuntimeError("sharp.cli.run_bakta was not found, but Bakta annotation is required.")

igem_pipeline = sharp_pipeline.igem_pipeline
run_bakta = sharp_cli.run_bakta

# ============================================================
# Resources from Cell 1
# ============================================================

heptamer_meme = Path(SHARP_INTERNAL_RESOURCES["heptamer_meme"]).resolve()
sarp_hmm = Path(SHARP_INTERNAL_RESOURCES["sarp_hmm"]).resolve()
domain_models_hmm = Path(SHARP_INTERNAL_RESOURCES["domain_models_hmm"]).resolve()
domain_modelnames = Path(SHARP_INTERNAL_RESOURCES["domain_modelnames"]).resolve()

for resource_path in [
    heptamer_meme,
    sarp_hmm,
    domain_models_hmm,
    domain_modelnames,
]:
    if not resource_path.exists() or resource_path.stat().st_size == 0:
        raise RuntimeError(f"Missing required resource: {resource_path}")

# ============================================================
# Prepare annotation inputs
# ============================================================

genome_nucleotide_fasta = ANNOTATION_DIR / "sharp_genome.fna"
genome_annotation = ANNOTATION_DIR / "sharp_annotation.gff3"
genome_protein_fasta = ANNOTATION_DIR / "sharp_proteins.faa"
genome_cds_fasta = ANNOTATION_DIR / "sharp_cds.ffn"

output_report = REPORT_DIR / "sharp_report.html"
ndf_table = TABLE_DIR / "sharp_neighborhoods.tsv"
fimo_table = TABLE_DIR / "sharp_fimo.tsv"
hmmscan_table = TABLE_DIR / "sharp_hmmscan.tsv"
summary_file = RUN_DIR / f"{RUN_NAME}_summary.json"
zip_path = RESULTS_DIR / f"{RUN_NAME}_sharp_results.zip"

shutil.copy2(GENOME_FASTA, genome_nucleotide_fasta)

if requires_bakta_execution:
    if not BAKTA_BIN:
        raise RuntimeError("Bakta is required but BAKTA_BIN is missing. Rerun Cell 1.")

    if not BAKTA_DB_PATH:
        raise RuntimeError("Bakta is required but BAKTA_DB_PATH is missing. Rerun Cell 1.")

    os.environ["BAKTA_DB"] = str(BAKTA_DB_PATH)

    bakta_output = BAKTA_RUN_DIR / "output"

    if bakta_output.exists():
        shutil.rmtree(bakta_output)

    bakta_output.mkdir(parents=True, exist_ok=True)

    bakta_extra = []
    skip_next = False

    for value in BAKTA_RUNTIME_ARGS:
        value = str(value)

        if skip_next:
            skip_next = False
            continue

        if value == "--threads":
            skip_next = True
            continue

        bakta_extra.append(value)

    print("Running Bakta annotation")

    bakta_result = run_bakta(
        str(genome_nucleotide_fasta),
        str(bakta_output),
        "sharp_bakta",
        str(BAKTA_DB_PATH),
        BAKTA_THREADS,
        bakta_extra,
    )

    if bakta_result is None:
        raise RuntimeError("sharp.cli.run_bakta returned None.")

    if len(bakta_result) < 3:
        raise RuntimeError("sharp.cli.run_bakta did not return annotation, protein FASTA, and nucleotide FASTA.")

    genome_annotation = Path(bakta_result[0]).resolve()
    genome_protein_fasta = Path(bakta_result[1]).resolve()
    genome_nucleotide_fasta = Path(bakta_result[2]).resolve()
    genome_format = "gff"

else:
    if ANNOTATION_FILE_FROM_CONTRACT is None or not ANNOTATION_FILE_FROM_CONTRACT.exists():
        raise RuntimeError("Annotated genome package mode requires an annotation file.")

    if PROTEIN_FASTA_FROM_CONTRACT is None or not PROTEIN_FASTA_FROM_CONTRACT.exists():
        raise RuntimeError("Annotated genome package mode requires a protein FASTA file.")

    genome_format = contract.get("genome_format", "")

    if not genome_format:
        suffix = ANNOTATION_FILE_FROM_CONTRACT.suffix.lower()
        genome_format = "gbk" if suffix in [".gb", ".gbk", ".gbff", ".genbank"] else "gff"

    if genome_format == "gbk":
        genome_annotation = ANNOTATION_DIR / "sharp_annotation.gbff"
    else:
        genome_annotation = ANNOTATION_DIR / "sharp_annotation.gff3"
        genome_format = "gff"

    shutil.copy2(ANNOTATION_FILE_FROM_CONTRACT, genome_annotation)
    shutil.copy2(PROTEIN_FASTA_FROM_CONTRACT, genome_protein_fasta)

    if CDS_FASTA_FROM_CONTRACT and CDS_FASTA_FROM_CONTRACT.exists():
        shutil.copy2(CDS_FASTA_FROM_CONTRACT, genome_cds_fasta)

# ============================================================
# Run official GitLab SHARP pipeline
# ============================================================

print(f"S(H)ARP GitLab pipeline started: {RUN_NAME}")

pipeline_result = igem_pipeline(
    genome_annotation=str(genome_annotation),
    genome_format=genome_format,
    genome_protein_fasta=str(genome_protein_fasta),
    genome_nucleotide_fasta=str(genome_nucleotide_fasta),
    models_path=[str(domain_models_hmm)],
    sarp_model=str(sarp_hmm),
    return_hmmscan=True,
    after=10,
    before=10,
    run_fimo=True,
    meme_file=str(heptamer_meme),
    return_fimo=True,
    make_figure=True,
    output_report=str(output_report),
    domains_filter=str(domain_modelnames),
    filter_mode="strict",
)

if not isinstance(pipeline_result, tuple) or len(pipeline_result) != 3:
    raise RuntimeError("sharp.pipeline.igem_pipeline did not return the expected tuple: ndf, fimo, hmmscan.")

ndf, fimo, hscan = pipeline_result

# ============================================================
# Export package outputs
# ============================================================

ndf.to_csv(ndf_table, sep="\t", index=False)
fimo.to_csv(fimo_table, sep="\t", index=False)
hscan.to_csv(hmmscan_table, sep="\t", index=False)

summary = {
    "status": "completed",
    "job_name": RUN_NAME,
    "input_mode": input_mode,
    "pipeline_package": "sharp",
    "pipeline_function": "sharp.pipeline.igem_pipeline",
    "sharp_file": str(Path(sharp.__file__).resolve()),
    "pipeline_file": str(Path(sharp_pipeline.__file__).resolve()),
    "genome_annotation": str(genome_annotation),
    "genome_format": genome_format,
    "genome_protein_fasta": str(genome_protein_fasta),
    "genome_nucleotide_fasta": str(genome_nucleotide_fasta),
    "models_path": [str(domain_models_hmm)],
    "sarp_model": str(sarp_hmm),
    "meme_file": str(heptamer_meme),
    "domains_filter": str(domain_modelnames),
    "ndf_rows": int(len(ndf)),
    "fimo_rows": int(len(fimo)),
    "hmmscan_rows": int(len(hscan)),
    "outputs": {
        "neighborhood_table": str(ndf_table),
        "fimo_table": str(fimo_table),
        "hmmscan_table": str(hmmscan_table),
        "html_report": str(output_report) if output_report.exists() else "",
        "zip": str(zip_path),
    },
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

summary_file.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

context.update({
    "pipeline_completed": True,
    "pipeline_package": "sharp",
    "pipeline_function": "sharp.pipeline.igem_pipeline",
    "summary_file": str(summary_file),
    "zip_path": str(zip_path),
    "html_report": str(output_report) if output_report.exists() else "",
    "summary": summary,
})

SHARP_CONTEXT_FILE.write_text(
    json.dumps(context, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(RUN_DIR))

globals().update({
    "NDF": ndf,
    "FIMO": fimo,
    "HMSCAN": hscan,
    "NDF_TABLE": ndf_table,
    "FIMO_TABLE": fimo_table,
    "HMSCAN_TABLE": hmmscan_table,
    "HTML_REPORT": output_report if output_report.exists() else None,
    "SUMMARY_FILE": summary_file,
    "ZIP_PATH": zip_path,
    "SHARP_ANALYSIS_SUMMARY": summary,
})

print("S(H)ARP GitLab pipeline complete")
print("ndf_rows:", len(ndf))
print("fimo_rows:", len(fimo))
print("hmmscan_rows:", len(hscan))
print("html_report:", output_report if output_report.exists() else "")
print("zip:", zip_path)

if DOWNLOAD_RESULTS_ZIP:
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        pass
